# Ayso 3D — صورة ← مجسّم

**١)** Runtime → Change runtime type → **T4 GPU** → Save
**٢)** Connect
**٣)** شغّل الخلية الوحيدة ▶ — تسوي كل شي (تحضير + رفع + توليد + تنزيل)

> كل شي بخلية وحدة — ما ينكسر لو انمسحت الذاكرة.

In [ ]:
# ═══════════════════════════════════════════════════════════
#  Ayso 3D — صورة ← مجسّم. خلية وحدة تسوي كل شي.
#  Runtime → Change runtime type → T4 GPU  (قبل التشغيل)
# ═══════════════════════════════════════════════════════════
import sys, os, subprocess, time, shutil

# ── ١) المكتبات + الكود ──
if not os.path.exists("/content/TripoSR/tsr"):
    print("تثبيت…")
    subprocess.run([sys.executable,"-m","pip","install","-q",
        "omegaconf","einops","trimesh","rembg","onnxruntime","scikit-image"], check=False)
    subprocess.run("cd /content && git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git",
                   shell=True, capture_output=True)
sys.path.insert(0, "/content/TripoSR")
sys.path.insert(0, "/content")

# بديل torchmcubes
if not os.path.exists("/content/torchmcubes.py"):
    open("/content/torchmcubes.py","w").write("""
import torch, numpy as np
from skimage import measure
def marching_cubes(vol, thresh, **kw):
    v = np.asarray(vol.detach().cpu().numpy() if hasattr(vol, "detach") else vol)
    if v.ndim == 4: v = v[0]
    v = v.astype(np.float32)
    if not np.isfinite(v).all() or v.min() >= thresh or v.max() <= thresh:
        return torch.zeros((0,3), dtype=torch.float32), torch.zeros((0,3), dtype=torch.int64)
    verts, faces, _, _ = measure.marching_cubes(v, level=float(thresh))
    return torch.from_numpy(verts.astype(np.float32)), torch.from_numpy(faces.astype(np.int64))
""")
import importlib; importlib.invalidate_caches()

import torch, torchmcubes
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download
from tsr.system import TSR
from tsr.utils import remove_background, resize_foreground
from PIL import Image
import numpy as np

assert torch.cuda.is_available(), "⚠️ فعّل T4: Runtime → Change runtime type"
print("كارت ✓")

# ── ٢) النموذج ──
print("تحميل النموذج (أول مرة ١.٧ غ)…")
cfg = OmegaConf.load(hf_hub_download("stabilityai/TripoSR", "config.yaml")); OmegaConf.resolve(cfg)
wgt = hf_hub_download("stabilityai/TripoSR", "model.ckpt")
MODEL = TSR(cfg)

sd = torch.load(wgt, map_location="cpu", weights_only=False)
R = [(".encoder.layer.",".layers."),
     (".attention.attention.query",".attention.q_proj"),
     (".attention.attention.key",".attention.k_proj"),
     (".attention.attention.value",".attention.v_proj"),
     (".attention.output.dense",".attention.o_proj"),
     (".intermediate.dense",".mlp.fc1"),
     (".output.dense",".mlp.fc2")]
nsd = {}
for k,v in sd.items():
    nk = k
    for a,b in R: nk = nk.replace(a,b)
    nsd[nk] = v
missing, unexpected = MODEL.load_state_dict(nsd, strict=False)
print(f"   ناقصة: {len(missing)} · زايدة: {len(unexpected)}")
crit = [k for k in missing if any(s in k for s in ("q_proj","k_proj","v_proj","o_proj","mlp.fc1","mlp.fc2"))]
assert not crit, f"أوزان ناقصة: {crit[:3]}"

# تحرير الذاكرة قبل التوليد (كولاب ١٢ غيغا رام فقط)
del sd, nsd
import gc; gc.collect()
MODEL.renderer.set_chunk_size(4096)
MODEL.to("cuda"); MODEL.eval()
torch.cuda.empty_cache()
gc.collect()
try:
    import psutil
    print(f"رام مستخدمة: {psutil.virtual_memory().percent}% · كارت: {torch.cuda.memory_allocated()/1e9:.2f} GB")
except Exception: pass
print("النموذج جاهز ✓")

# ── ٣) الصورة ──
from google.colab import files
print("\nارفع صورة:")
up = files.upload()
IMAGE = "/content/" + list(up.keys())[0]
im0 = Image.open(IMAGE)
print("الصورة:", im0.size)
display(im0.convert("RGB").resize((260, int(260*im0.size[1]/im0.size[0]))))

# ── ٤) التوليد ──
img = Image.open(IMAGE).convert("RGBA")
if np.array(img)[:,:,3].max() == 0:
    try:
        import rembg
        img = rembg.remove(img, session=rembg.new_session())
        print("خلفية انشالت ✓")
    except Exception as e:
        print("rembg فشل:", str(e)[:50])

proc = resize_foreground(remove_background(img), 0.85)
a = np.array(proc.convert("RGB")).astype(np.float32)/255.0
a = a * np.array(proc.convert("RGBA"))[:,:,3:4].astype(np.float32)/255.0

RES = 128   # 128 آمن على T4 · 256 أدق بس قد ينفد الرام
gc.collect(); torch.cuda.empty_cache()
print(f"التوليد (دقة {RES})…")
t0 = time.time()
with torch.no_grad():
    codes = MODEL(torch.from_numpy(a).cuda(), device="cuda")   # (H,W,C) مباشرة — بلا list
    mesh = MODEL.extract_mesh(codes, has_vertex_color=True, resolution=RES)[0]
gc.collect(); torch.cuda.empty_cache()
print(f"خلص بـ {time.time()-t0:.1f} ثانية")
print(f"رؤوس {len(mesh.vertices):,} · وجوه {len(mesh.faces):,}")

mesh.export("/content/model.glb")
mesh.export("/content/model.obj")
print(f"model.glb · {os.path.getsize('/content/model.glb')/1e6:.1f} MB")

# ── ٥) عرض ٣ زوايا ──
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
V = np.asarray(mesh.vertices); F = np.asarray(mesh.faces)
if len(F) > 3000: F = F[np.random.choice(len(F), 3000, replace=False)]
fig = plt.figure(figsize=(15,5), facecolor="white")
for i,(el,az,nm) in enumerate([(18,35,"أمامي"),(0,0,"جانبي"),(75,20,"علوي")]):
    ax = fig.add_subplot(1,3,i+1, projection="3d")
    pc = Poly3DCollection(V[F], alpha=1.0); pc.set_facecolor("#4a90d9"); pc.set_edgecolor("none")
    ax.add_collection3d(pc); ax.auto_scale_xyz(V[:,0],V[:,1],V[:,2])
    ax.set_box_aspect((1,1,1)); ax.view_init(el,az); ax.set_axis_off(); ax.set_title(nm, fontsize=14)
plt.tight_layout()
plt.savefig("/content/samples.png", dpi=100, facecolor="white", bbox_inches="tight")
plt.show()

# ── ٦) تنزيل ──
subprocess.run("cd /content && zip -q -j ayso-3d.zip model.glb model.obj samples.png", shell=True)
print("\nالحجم:", round(os.path.getsize("/content/ayso-3d.zip")/1e6,1), "MB")
files.download("/content/ayso-3d.zip")
files.download("/content/model.glb")
print("✅ خلص")